In [ ]:
import pandas as pd
import requests

url = "https://markets.newyorkfed.org/api/rates/secured/all/search.json"
params = {'startDate': '2000-01-01' , 'endDate': '2026-08-05'}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()

    df = pd.DataFrame(data)

else:
    print(f'Error: {response.status_code}')

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df['refRates'][5], df['refRates'][6], df['refRates'][7]

In [ ]:
df['refRates'].str['type'].value_counts()

In [ ]:
flat = pd.json_normalize(df['refRates'])

In [ ]:
flat.tail(10)

In [ ]:
out = (flat
       .assign(n=flat.groupby('effectiveDate').cumcount())
       .pivot(index='effectiveDate', columns='n'))
out.columns = [f'{a}_{b}' for a, b in out.columns]
out = out.reset_index()

In [ ]:
out['effectiveDate'] = pd.to_datetime(out['effectiveDate'])

In [ ]:
out.head(10)

In [ ]:
out.head(1)

In [ ]:
out.columns

In [ ]:
cols = ['percentRate_0', 'percentRate_1', 'percentRate_2', 'percentRate_3',
       'percentPercentile1_0', 'percentPercentile1_1', 'percentPercentile1_2',
       'percentPercentile1_3', 'percentPercentile25_0',
       'percentPercentile25_1', 'percentPercentile25_2',
       'percentPercentile25_3', 'percentPercentile75_0',
       'percentPercentile75_1', 'percentPercentile75_2',
       'percentPercentile75_3', 'percentPercentile99_0',
       'percentPercentile99_1', 'percentPercentile99_2',
       'percentPercentile99_3', 'volumeInBillions_0', 'volumeInBillions_1',
       'volumeInBillions_2', 'volumeInBillions_3',
       'average30day_0', 'average30day_1', 'average30day_2', 'average30day_3',
       'average90day_0', 'average90day_1', 'average90day_2', 'average90day_3',
       'average180day_0', 'average180day_1', 'average180day_2',
       'average180day_3']

out[cols] = out[cols].apply(pd.to_numeric, errors='coerce')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.figure(figsize=(15, 4))

plt.plot(out['effectiveDate'], out['percentRate_2'])

In [ ]:
dates = out['effectiveDate']
value0 = out['percentRate_0']
value1 = out['percentRate_1']
value2 = out['percentRate_2']

plt.figure(figsize=(15, 4))

plt.plot(dates, value0, color = 'red', label = 'TGCR', linestyle = '-')
plt.plot(dates, value1, color = 'green', label = 'BGCR', linestyle = '--')
plt.plot(dates, value2, color = 'blue', label = 'SOFR', linestyle = '-.')

In [ ]:
diffvalue = value2 - value0

plt.figure(figsize=(15, 4))

plt.plot(dates, diffvalue, color = 'red')


In [ ]:
diffvalue.mean()

In [ ]:
df['refRates'][0], df['refRates'][1]

In [ ]:
out[out['percentPercentile99_2'] == 'NA']

In [ ]:
filteredout = out[out['effectiveDate'] >= '2020-06-01']

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(filteredout['effectiveDate'], filteredout['percentPercentile99_2'] - filteredout['percentPercentile1_2'])

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(filteredout['effectiveDate'], filteredout['percentPercentile99_2'] - filteredout['percentRate_2'])

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(filteredout['effectiveDate'], filteredout['percentPercentile1_2'] - filteredout['percentRate_2'])

In [ ]:
#SOFR Tail spread 

plt.figure(figsize=(15, 10))

plt.plot(filteredout['effectiveDate'], filteredout['percentPercentile1_2'] - filteredout['percentRate_2'])
plt.plot(filteredout['effectiveDate'], filteredout['percentPercentile99_2'] - filteredout['percentRate_2'])



In [ ]:
#SOFR99 - BGCR99

plt.figure(figsize=(15,5))

plt.plot(out['effectiveDate'], out['percentPercentile99_2'] - out['percentPercentile99_1'])

In [ ]:
#BGCR99 - TGCR99

plt.figure(figsize=(15,5))

plt.plot(out['effectiveDate'], out['percentPercentile99_1'] - out['percentPercentile99_0'])

In [ ]:
#SOFR - BGCR

plt.figure(figsize=(15,5))

plt.plot(out['effectiveDate'], out['percentRate_2'] - out['percentRate_1'])

In [ ]:
#BGCR - TGCR

plt.figure(figsize=(15,5))

plt.plot(out['effectiveDate'], out['percentRate_1'] - out['percentRate_0'])

In [ ]:
#SOFR tail dispersion


plt.figure(figsize=(15, 10))

plt.plot(out['effectiveDate'], out['percentPercentile1_2'] - out['percentRate_2'])
plt.plot(out['effectiveDate'], out['percentPercentile99_2'] - out['percentRate_2'])



In [ ]:
#BGCR tail dispersion


plt.figure(figsize=(15, 10))

plt.plot(out['effectiveDate'], out['percentPercentile1_1'] - out['percentRate_1'])
plt.plot(out['effectiveDate'], out['percentPercentile99_1'] - out['percentRate_1'])


In [ ]:
#TGCR tail dispersion


plt.figure(figsize=(15, 10))

plt.plot(out['effectiveDate'], out['percentPercentile1_0'] - out['percentRate_0'])
plt.plot(out['effectiveDate'], out['percentPercentile99_0'] - out['percentRate_0'])
